In [1]:
#!pip install geopandas pandas shapely

In [2]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Define the base directory (Uncomment the active path)
base_path = '..' # Currently active path

# Input paths
shapefile_path = os.path.join(base_path, 'includes', 'maps', 'itapua', 'Mapa_Itapua.shp')
consumers_csv_path = os.path.join(base_path, 'includes', 'Tabela_consumidores_Itapua_convertida.csv')

# Output path
output_csv_path = os.path.join(base_path, 'includes', 'Tabela_consumidores_Itapua_com_setor.csv')

# ==========================================
# 2. Load Census Tracts (Shapefile)
# ==========================================

# Load the shapefile
census_tracts = gpd.read_file(shapefile_path)

# Select only the Sector ID (CD_SETOR) and geometry columns
census_tracts = census_tracts[['CD_SETOR', 'geometry']]

# Display the first few rows to verify loading
#print("Census Tracts loaded:")
#print(census_tracts.head())

# ==========================================
# 3. Load Consumer Data (CSV)
# ==========================================

# Load the CSV file containing supply points
# separator is set to ';' as per original file structure
consumers_df = pd.read_csv(consumers_csv_path, sep=';')
original_count = len(consumers_df)

#print("\nConsumer Data loaded:")
#print(consumers_df.head())

# ==========================================
# 4. Convert to GeoDataFrame
# ==========================================

# Create Point geometries from Longitude and Latitude columns
# Note: Ensure the columns 'LONG_GEO' and 'LAT_GEO' exist in your CSV
geometry = [Point(xy) for xy in zip(consumers_df['LONG_GEO'], consumers_df['LAT_GEO'])]

# Initialize GeoDataFrame
# EPSG:4326 is the standard WGS84 coordinate system (Lat/Lon)
consumers_gdf = gpd.GeoDataFrame(consumers_df, geometry=geometry, crs="EPSG:4326")

# ==========================================
# 5. Spatial Join
# ==========================================

# Reproject consumer points to match the CRS of the census tracts shapefile
consumers_gdf = consumers_gdf.to_crs(census_tracts.crs)

# Perform Spatial Join (Left Join)
# 'predicate="within"' checks if the point falls inside the polygon
spatial_join_result = gpd.sjoin(consumers_gdf, census_tracts, how="left", predicate="within")

# Drop unnecessary columns generated by the join
spatial_join_result.drop(columns=['geometry', 'index_right'], inplace=True)

# Remove points that fell outside the target census sectors (if any)
outside_count = spatial_join_result['CD_SETOR'].isna().sum()

# If sector not found use 00000000000
spatial_join_result['CD_SETOR'] = spatial_join_result['CD_SETOR'].fillna("00000000000")
#spatial_join_result.dropna(subset=['CD_SETOR'], inplace=True)

# ==========================================
# 6. Save Results
# ==========================================

# Save the enriched dataframe to a new CSV file
spatial_join_result.to_csv(output_csv_path, index=False)

print("-" * 40)
print(f"Processing complete.")
print(f"Original records: {original_count}")
print(f"Current records (Saved): {len(spatial_join_result)}")
print(f"Records mapped to sectors: {original_count - outside_count}")
print(f"Records outside sectors (kept): {outside_count}")
print(f"File saved to: {output_csv_path}")
print("-" * 40)
    
print("Sample output:")
print(spatial_join_result.head())

----------------------------------------
Processing complete.
Original records: 19630
Current records (Saved): 19630
Records mapped to sectors: 18599
Records outside sectors (kept): 1031
File saved to: ..\includes\Tabela_consumidores_Itapua_com_setor.csv
----------------------------------------
Sample output:
                                        SK_MATRICULA NM_LOCALIDADE  \
0  00F7CB2CF5D05D0F776C806909857017EE517B778064A0...  SALVADOR UMB   
1  0234239DB83A648AEDDEE894268C57F481137150104260...  SALVADOR UMB   
2  02386243BC53EC23E402783573B199C090EE1F970CDC06...  SALVADOR UMB   
3  02F8E4CC0BE2ACD019A75698AC85C0EF477C6A3A2EC274...  SALVADOR UMB   
4  031B7CB19D864622172D64F0B4FE03BEBF93DF9F3C81F1...  SALVADOR UMB   

  NM_CATEGORIATARIFARIA NM_SITUACAO_IMOVEL  NN_MORADORES  ST_PISCINA  \
0           RESIDENCIAL           HABITADO             0           0   
1           RESIDENCIAL           HABITADO             0           0   
2           RESIDENCIAL           HABITADO          